# Step 2: Train SAE on Saved Activations
Loads the `.pt` activation chunks saved in Step 1 and trains a Sparse Autoencoder.
No VLM needed in memory — just pure tensor training.

In [1]:
!git clone https://github.com/ExplainableML/sae-for-vlm.git /kaggle/working/sae-for-vlm 2>/dev/null 
!git clone https://github.com/saprmarks/dictionary_learning.git /kaggle/working/dictionary_learning 2>/dev/null
!pip install -q -r /kaggle/working/sae-for-vlm/requirements.txt 2>/dev/null
!pip install -q nnsight

# Add both repos to path
import sys
sys.path.insert(0, "/kaggle/working/sae-for-vlm")
sys.path.insert(0, "/kaggle/working/dictionary_learning")
print("Paths added.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.5/164.5 kB 5.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 6.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.8/170.8 kB 11.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.8/60.8 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.7/183.7 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.8/59.8 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.1/82.1 kB 6.5 MB/s eta 0:00:00
Paths added.


In [2]:
# ── AUTHENTICATION: W&B AND HUGGING FACE ─────────────────────────────────────
import os
import wandb
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

print("Fetching credentials from Kaggle Secrets...")
user_secrets = UserSecretsClient()

# W&B Login
wandb_key = user_secrets.get_secret("WANDB_API_KEY")
wandb.login(key=wandb_key)
print("Logged into Weights & Biases")

# Hugging Face Login
hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token)
print("Logged into Hugging Face")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


Fetching credentials from Kaggle Secrets...


wandb: Currently logged in as: gayushkatni (gayushkatni-iiit-hyderabad) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Logged into Weights & Biases
Logged into Hugging Face


In [3]:
import os
import glob
import torch
from torch.utils.data import Dataset, DataLoader

# ── CONFIG ──────────────────────────────────────────────────────────────────
TRAIN_DIR        = "/kaggle/input/datasets/akgiiith/clip-v14-activations/train"
VAL_DIR          = "/kaggle/input/datasets/akgiiith/clip-v14-activations/val"
CHECKPOINT_DIR   = "/kaggle/working/sae_checkpoints"

SAE_MODEL        = "batch_top_k"   # batch_top_k, matroyshka_batch_top_k, top_k, standard, jumprelu
EXPANSION_FACTOR = 4               # dict_size = expansion_factor × activation_dim (1024 → 4096 features)
K                = 20              # Number of active features per sample (paper uses 20)
BATCH_SIZE       = 4096            # SAE training batch size
STEPS            = 100_000         # Total training steps
SAVE_STEPS       = 5_000           # Save checkpoint every N steps
LOG_STEPS        = 100             # Print loss every N steps
DEVICE           = "cuda" if torch.cuda.is_available() else "cpu"
USE_WANDB        = True         # Set True if you have a wandb account

# LR formula from paper: 16 / (125 * sqrt(dict_size))
import math
ACTIVATION_DIM   = 1024            # CLIP ViT-L/14 hidden dim
DICT_SIZE        = EXPANSION_FACTOR * ACTIVATION_DIM
LR               = 16 / (125 * math.sqrt(DICT_SIZE))

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f"Device:         {DEVICE}")
print(f"SAE type:       {SAE_MODEL}")
print(f"Activation dim: {ACTIVATION_DIM}")
print(f"Dictionary size:{DICT_SIZE}  (expansion ×{EXPANSION_FACTOR})")
print(f"K (sparsity):   {K}")
print(f"Learning rate:  {LR:.6f}")
print(f"Steps:          {STEPS:,}")

Device:         cuda
SAE type:       batch_top_k
Activation dim: 1024
Dictionary size:4096  (expansion ×4)
K (sparsity):   20
Learning rate:  0.002000
Steps:          100,000


In [4]:
# ── DATASET: reads all .pt chunk files ──────────────────────────────────────
class ActivationsDataset(Dataset):
    """
    Reads all chunk_XXXXX.pt files from a directory.
    Each file is a tensor of shape [N, activation_dim].
    Concatenates them all into one big tensor in memory.
    If total activations are too large for RAM, reduce MAX_IMAGES in Step 1.
    """
    def __init__(self, directory):
        chunk_files = sorted(glob.glob(os.path.join(directory, "*.pt")))
        if not chunk_files:
            raise FileNotFoundError(f"No .pt files found in {directory}")
        print(f"Loading {len(chunk_files)} chunk files from {directory}...")
        chunks = [torch.load(f) for f in chunk_files]
        self.data = torch.cat(chunks, dim=0).float()  # [total_N, dim]
        print(f"  → {self.data.shape[0]:,} activation vectors, dim={self.data.shape[1]}")

    def __len__(self):
        return self.data.shape[0]

    def __getitem__(self, idx):
        return self.data[idx]


train_dataset = ActivationsDataset(TRAIN_DIR)
val_dataset   = ActivationsDataset(VAL_DIR)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
print("DataLoaders ready.")

Loading 71 chunk files from /kaggle/input/datasets/akgiiith/clip-v14-activations/train...
  → 45,000 activation vectors, dim=1024
Loading 8 chunk files from /kaggle/input/datasets/akgiiith/clip-v14-activations/val...
  → 5,000 activation vectors, dim=1024
DataLoaders ready.


In [5]:
# ── BUILD TRAINER CONFIG ─────────────────────────────────────────────────────
from dictionary_learning.trainers.batch_top_k import BatchTopKTrainer
from dictionary_learning.trainers.matryoshka_batch_top_k import MatryoshkaBatchTopKTrainer
from dictionary_learning.trainers.top_k import TopKTrainer
from dictionary_learning.trainers.standard import StandardTrainer
from dictionary_learning.trainers.jumprelu import JumpReluTrainer
from dictionary_learning.training import trainSAE

TRAINER_MAP = {
    "batch_top_k":             BatchTopKTrainer,
    "matroyshka_batch_top_k":  MatryoshkaBatchTopKTrainer,
    "top_k":                   TopKTrainer,
    "standard":                StandardTrainer,
    "jumprelu":                JumpReluTrainer,
}

trainer_cfg = {
    "trainer":          TRAINER_MAP[SAE_MODEL],
    "activation_dim":   ACTIVATION_DIM,
    "dict_size":        DICT_SIZE,
    "lr":               LR,
    "device":           DEVICE,
    "steps":            STEPS,
    "layer":            "22",          # metadata only — for saving in config.json
    "lm_name":          "clip-vit-l-14",
    "submodule_name":   "vision_encoder",
    "k":                K,
    "auxk_alpha":       1/32,
    "decay_start":      int(STEPS * 0.8),  # decay LR in last 20% of training
    "threshold_beta":   0.999,
    "threshold_start_step": 1000,
}

# Add Matryoshka group fractions if using that variant
if SAE_MODEL == "matroyshka_batch_top_k":
    trainer_cfg["group_fractions"] = [0.0625, 0.1875, 0.4375, 1.0]

print("Trainer config:")
for k_name, v in trainer_cfg.items():
    if k_name != "trainer":
        print(f"  {k_name}: {v}")

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


Trainer config:
  activation_dim: 1024
  dict_size: 4096
  lr: 0.002
  device: cuda
  steps: 100000
  layer: 22
  lm_name: clip-vit-l-14
  submodule_name: vision_encoder
  k: 20
  auxk_alpha: 0.03125
  decay_start: 80000
  threshold_beta: 0.999
  threshold_start_step: 1000


In [6]:
# ── TRAIN ────────────────────────────────────────────────────────────────────
print("\nStarting SAE training...")
print(f"Total activations in training set: {len(train_dataset):,}")
print(f"Batches per epoch: {len(train_loader):,}")
print(f"Training for {STEPS:,} steps (~{STEPS / len(train_loader):.1f} epochs)\n")

save_steps_list = list(range(0, STEPS, SAVE_STEPS))

# Helper: Infinite Data Generator to handle Epochs & Device Moving
def get_infinite_batches(dataloader, device):
    """Loops over the dataloader forever and pushes batches to the GPU."""
    while True:
        for batch in dataloader:
            yield batch.to(device)

# Initialize the generator on the correct device
train_data_gen = get_infinite_batches(train_loader, DEVICE)

ae = trainSAE(
    data=train_data_gen,           
    trainer_configs=[trainer_cfg.copy()],
    steps=STEPS,
    save_steps=save_steps_list,
    save_dir=CHECKPOINT_DIR,
    log_steps=LOG_STEPS,
    use_wandb=USE_WANDB,
)

print(f"\nTraining complete! SAE saved to {CHECKPOINT_DIR}")


Starting SAE training...
Total activations in training set: 45,000
Batches per epoch: 11
Training for 100,000 steps (~9090.9 epochs)



  0%|          | 0/100000 [00:00<?, ?it/s]Process Process-1:
Traceback (most recent call last):
  File "/usr/lib/python3.12/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/usr/lib/python3.12/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/kaggle/working/dictionary_learning/dictionary_learning/training.py", line 23, in new_wandb_process
    wandb.init(entity=entity, project=project, config=config, name=config["wandb_name"])
  File "/usr/local/lib/python3.12/dist-packages/wandb/sdk/wandb_init.py", line 1595, in init
    get_sentry().reraise(e)
  File "/usr/local/lib/python3.12/dist-packages/wandb/analytics/sentry.py", line 190, in reraise
    raise exc.with_traceback(tb)
  File "/usr/local/lib/python3.12/dist-packages/wandb/sdk/wandb_init.py", line 1578, in init
    run = wi.init(run_settings, run_config, run_printer)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12


Training complete! SAE saved to /kaggle/working/sae_checkpoints


In [7]:
# ── QUICK SANITY CHECK ───────────────────────────────────────────────────────
print("=== Quick Sanity Check ===")

# Load a batch of val activations
sample_batch = next(iter(val_loader)).to(DEVICE)
print(f"Input batch shape: {sample_batch.shape}")

# Move SAE to device and eval
ae = ae.to(DEVICE)
ae.eval()

with torch.no_grad():
    recon, features = ae(sample_batch, output_features=True)

# Reconstruction quality
mse = torch.mean((sample_batch - recon) ** 2).item()
var = torch.var(sample_batch).item()
r2  = 1 - mse / var

# Sparsity
l0 = (features > 0).float().sum(dim=-1).mean().item()

print(f"\nReconstruction R²:  {r2:.4f}  (closer to 1.0 is better)")
print(f"Mean active features (L0): {l0:.1f}  (target ≈ K={K})")
print(f"MSE loss: {mse:.6f}")
print(f"\nFeature activation stats:")
print(f"  Max:  {features.max().item():.4f}")
print(f"  Mean: {features.mean().item():.6f}")
print(f"  % dead (never activated in batch): {(features.sum(0) == 0).float().mean().item()*100:.1f}%")

=== Quick Sanity Check ===
Input batch shape: torch.Size([4096, 1024])


AttributeError: 'NoneType' object has no attribute 'to'

In [ ]:
# ── INSPECT TOP ACTIVATING FEATURES (visualization prep) ────────────────────
# For each neuron, find which samples in the val batch activate it most.
# This is a tiny preview — real visualization uses visualize_neurons.py

print("=== Top 5 neurons by max activation in this batch ===")
max_per_neuron = features.max(dim=0).values  # [dict_size]
top5_neurons = max_per_neuron.topk(5).indices.tolist()

for neuron_idx in top5_neurons:
    acts = features[:, neuron_idx]
    nonzero = (acts > 0).sum().item()
    print(f"  Neuron #{neuron_idx:5d}: max_act={acts.max().item():.3f},  active on {nonzero}/{len(acts)} samples")

print("\n✓ SAE is working. Proceed to metric.py for MonoSemanticity Score computation.")